# 🫁 VisionCare AI: Multi-Label Training, Calibration & Explainability Benchmark
**Project:** VisionCare AI (Explainable AI-Powered Radiology Assistant)  
**Dataset:** NIH ChestX-ray14 (`nih-chest-xrays/data`)  
**Hardware:** Kaggle GPU (NVIDIA T4 x2 or P100) — *Ensure Accelerator is set to GPU and Internet is ON*

---

### Pipeline Objectives:
1. **Patient-Isolated Multi-Label Stratification:** Extract 25,000 images for training/val, while strictly keeping the 983 radiologist bounding-box images in an unseen holdout set.
2. **DenseNet-121 Transfer Learning:** Train on $224 \times 224$ images via 2-stage fine-tuning (warmup classifier head $\to$ fine-tune `denseblock4` + `norm5`).
3. **Confidence Calibration:** Post-hoc Temperature Scaling via L-BFGS to optimize per-class temperatures $T_i$ minimizing Expected Calibration Error (ECE).
4. **Operating Thresholds:** Youden's J-statistic sensitivity/specificity optimization.
5. **Explainability Benchmarking:** Grad-CAM++ evaluation against 1,600 radiologist bounding boxes ($IoBB$ and $IoU$).
6. **Artifact Export:** Save 4 deployment files to `/kaggle/working/` ready to download for the local workstation backend.


### Step 0: Imports, Environment Setup & 14-Disease Taxonomy


In [ ]:
import os
import sys
import json
import glob
import time
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW, lr_scheduler
import torch.optim as optim

from sklearn.metrics import roc_auc_score, roc_curve

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Training Device: {device}")
if torch.cuda.is_available():
    print(f"[*] GPU Name: {torch.cuda.get_device_name(0)}")

# NIH ChestX-ray14 Taxonomy
DISEASE_CLASSES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"
]
NUM_CLASSES = len(DISEASE_CLASSES)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f"[+] Loaded {NUM_CLASSES} Target Pathologies:")
print("   ", ", ".join(DISEASE_CLASSES))


### Step 1: Discover Dataset & Index Images Across Archives
Kaggle partitions 112,120 PNGs across multiple subfolders (`images_001` .. `images_012`). We construct an in-memory hash map `{filename: absolute_disk_path}` to enable instant $O(1)$ lookups.


In [ ]:
def locate_kaggle_dataset() -> Tuple[str, str, str]:
    data_dir = "/kaggle/input/data" if os.path.exists("/kaggle/input/data") else "/kaggle/input/nih-chest-xrays/data"
    
    entry_candidates = [
        os.path.join(data_dir, "Data_Entry_2017.csv"),
        os.path.join(data_dir, "Data_Entry_2017_v2020.csv"),
        "Data_Entry_2017.csv"
    ]
    data_entry_csv = next((c for c in entry_candidates if os.path.exists(c)), "")
    
    bbox_candidates = [
        os.path.join(data_dir, "BBox_List_2017.csv"),
        "BBox_List_2017.csv"
    ]
    bbox_csv = next((c for c in bbox_candidates if os.path.exists(c)), "")
    
    return data_dir, data_entry_csv, bbox_csv

data_dir, data_entry_csv, bbox_csv = locate_kaggle_dataset()
print(f"[+] Data Directory: {data_dir}")
print(f"[+] Metadata CSV:   {data_entry_csv}")
print(f"[+] Bounding Boxes: {bbox_csv}")

# Build in-memory image lookup table
print("[*] Scanning image directories ...")
image_lookup = {}
patterns = [
    os.path.join(data_dir, "images*", "images", "*.png"),
    os.path.join(data_dir, "images*", "*.png"),
    os.path.join(data_dir, "*.png")
]
for pattern in patterns:
    for filepath in glob.glob(pattern):
        fname = os.path.basename(filepath)
        if fname not in image_lookup:
            image_lookup[fname] = filepath

print(f"[+] Successfully indexed {len(image_lookup)} images.")


### Step 2: Patient-Isolated Multi-Label Stratification
**Critical Clinical ML Rule:** All longitudinal studies from the same patient must reside strictly within Train OR Val. Additionally, we isolate the **983 images** from `BBox_List_2017.csv` into an unseen test set for explainability benchmarking.


In [ ]:
df = pd.read_csv(data_entry_csv)
bbox_df = pd.read_csv(bbox_csv) if bbox_csv and os.path.exists(bbox_csv) else None

# 1. Isolate BBox Holdout
if bbox_df is not None:
    bbox_image_indices = set(bbox_df["Image Index"].unique())
    holdout_df = df[df["Image Index"].isin(bbox_image_indices)].copy()
    candidate_df = df[~df["Image Index"].isin(bbox_image_indices)].copy()
    print(f"[+] Isolated {len(holdout_df)} images for explainability evaluation.")
else:
    holdout_df = pd.DataFrame()
    candidate_df = df.copy()

# 2. Sample 25,000 images grouped by Patient ID
SAMPLE_SIZE = 25000
VAL_RATIO = 0.15
np.random.seed(42)

unique_patients = candidate_df["Patient ID"].unique()
np.random.shuffle(unique_patients)

selected_patients = []
collected_images = 0
patient_counts = candidate_df.groupby("Patient ID")["Image Index"].count().to_dict()

for pid in unique_patients:
    selected_patients.append(pid)
    collected_images += patient_counts.get(pid, 1)
    if collected_images >= SAMPLE_SIZE:
        break

stratified_df = candidate_df[candidate_df["Patient ID"].isin(selected_patients)].copy()
unique_strat_patients = np.array(selected_patients)

num_val_patients = int(len(unique_strat_patients) * VAL_RATIO)
val_patients = set(unique_strat_patients[:num_val_patients])
train_patients = set(unique_strat_patients[num_val_patients:])

train_df = stratified_df[stratified_df["Patient ID"].isin(train_patients)].copy().reset_index(drop=True)
val_df   = stratified_df[stratified_df["Patient ID"].isin(val_patients)].copy().reset_index(drop=True)

print(f"[+] Train Set: {len(train_df)} images across {len(train_patients)} unique patients")
print(f"[+] Val Set:   {len(val_df)} images across {len(val_patients)} unique patients")


### Step 3: High-Performance PyTorch Dataset ($224 \times 224$)
Images are resized to $224 \times 224$ on-the-fly and normalized with ImageNet statistics. Labels are converted into 14-dimensional binary multi-hot vectors (`Finding Labels.split('|')`).


In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_lookup: Dict[str, str], target_size=(224, 224), is_training=False):
        self.df = df.reset_index(drop=True)
        self.image_lookup = image_lookup
        self.target_size = target_size
        self.is_training = is_training

        # Precompute 14-dimensional multi-hot binary label matrix
        self.labels = np.zeros((len(self.df), NUM_CLASSES), dtype=np.float32)
        for idx, finding_str in enumerate(self.df["Finding Labels"]):
            findings = str(finding_str).split("|")
            for c_idx, disease in enumerate(DISEASE_CLASSES):
                if disease in findings:
                    self.labels[idx, c_idx] = 1.0

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, str]:
        fname = self.df.iloc[idx]["Image Index"]
        img_path = self.image_lookup.get(fname, "")

        if img_path and os.path.exists(img_path):
            img = Image.open(img_path).convert("RGB")
        else:
            img = Image.new("RGB", self.target_size, color=(128, 128, 128))

        img = img.resize(self.target_size, Image.BILINEAR)
        img_arr = np.array(img, dtype=np.float32) / 255.0

        if self.is_training:
            # Mild horizontal flip
            if np.random.rand() > 0.5:
                img_arr = np.fliplr(img_arr).copy()

        img_tensor = torch.from_numpy(img_arr.copy()).permute(2, 0, 1).float()
        for c in range(3):
            img_tensor[c] = (img_tensor[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]

        label_tensor = torch.from_numpy(self.labels[idx]).float()
        return img_tensor, label_tensor, fname

BATCH_SIZE = 32
train_ds = ChestXrayDataset(train_df, image_lookup, is_training=True)
val_ds   = ChestXrayDataset(val_df, image_lookup, is_training=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"[+] DataLoaders ready: Train batches = {len(train_loader)} | Val batches = {len(val_loader)}")


### Step 4: Model Initialization & Multi-Label Loss with Class Weighting
We initialize DenseNet121 with ImageNet weights and replace the classifier head with `nn.Linear(1024, 14)`. To combat extreme medical class imbalance (e.g. Hernia $< 1\%$, Infiltration $> 20\%$), we compute positive class weights.


In [ ]:
import torchvision.models as models

# Initialize DenseNet121 with ImageNet pretrained weights
try:
    weights = models.DenseNet121_Weights.IMAGENET1K_V1
    model = models.densenet121(weights=weights)
except AttributeError:
    model = models.densenet121(pretrained=True)

# Replace classifier head for 14 pathologies
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

# Compute positive class weights for BCEWithLogitsLoss
pos_counts = train_ds.labels.sum(axis=0)
neg_counts = len(train_ds) - pos_counts
pos_weights = torch.tensor(
    [(neg_counts[i] + 1) / (pos_counts[i] + 1) for i in range(NUM_CLASSES)],
    device=device
)
# Clamp weights between 1.0 and 15.0 to maintain training stability
pos_weights = torch.clamp(pos_weights, min=1.0, max=15.0)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

print("[+] DenseNet-121 initialized with ImageNet weights and adapted for 14 classes.")
print("[+] Positive loss weights calculated per disease class.")


### Step 5: Two-Stage Transfer Learning Loop
* **Stage 1 (Warmup):** Freeze `features`, train only the 14-class classifier head for 2 epochs ($lr = 10^{-3}$).
* **Stage 2 (Fine-Tuning):** Unfreeze `denseblock4` and `norm5`, train for 6 epochs with Cosine Annealing ($lr = 10^{-4} \to 10^{-6}$).


In [ ]:
# ==============================================================================
# STAGE 1: Warmup Classifier Head (Backbone Frozen)
# ==============================================================================
print("=" * 70)
print("STAGE 1: Warmup Classifier Head (2 Epochs)")
print("=" * 70)

for param in model.features.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

optimizer_head = AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(2):
    model.train()
    running_loss = 0.0
    start_t = time.time()
    for imgs, labels, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_head.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer_head.step()
        running_loss += loss.item()
    
    elapsed = time.time() - start_t
    print(f"[*] Stage 1 - Epoch [{epoch+1}/2] - Loss: {running_loss/len(train_loader):.4f} ({elapsed:.1f}s)")

# ==============================================================================
# STAGE 2: Fine-Tuning denseblock4 + norm5 + Classifier
# ==============================================================================
print("\n" + "=" * 70)
print("STAGE 2: Fine-Tuning denseblock4 + norm5 (6 Epochs)")
print("=" * 70)

for name, param in model.features.named_parameters():
    if "denseblock4" in name or "norm5" in name:
        param.requires_grad = True

optimizer_full = AdamW([
    {"params": model.features.denseblock4.parameters(), "lr": 1e-4},
    {"params": model.features.norm5.parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 5e-4}
], weight_decay=1e-4)

scheduler = lr_scheduler.CosineAnnealingLR(optimizer_full, T_max=6, eta_min=1e-6)

best_val_auc = 0.0
os.makedirs("/kaggle/working/pretrained", exist_ok=True)
best_checkpoint_path = "/kaggle/working/pretrained/densenet121_nih14_best.pth"

for epoch in range(6):
    model.train()
    running_loss = 0.0
    start_t = time.time()
    for imgs, labels, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_full.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer_full.step()
        running_loss += loss.item()
    
    scheduler.step()
    elapsed = time.time() - start_t

    # Quick validation pass
    model.eval()
    val_logits_list, val_labels_list = [], []
    with torch.no_grad():
        for imgs, labels, _ in val_loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            val_logits_list.append(logits.cpu())
            val_labels_list.append(labels)
            
    val_logits = torch.cat(val_logits_list, dim=0)
    val_labels = torch.cat(val_labels_list, dim=0)
    val_probs  = torch.sigmoid(val_logits).numpy()
    val_targets = val_labels.numpy()

    aucs = []
    for i in range(NUM_CLASSES):
        if np.sum(val_targets[:, i]) > 0:
            try:
                auc = roc_auc_score(val_targets[:, i], val_probs[:, i])
                aucs.append(auc)
            except ValueError:
                pass
    mean_auc = np.mean(aucs) if aucs else 0.50
    print(f"[*] Stage 2 - Epoch [{epoch+1}/6] - Loss: {running_loss/len(train_loader):.4f} | Val Mean AUROC: {mean_auc:.4f} ({elapsed:.1f}s)")

    if mean_auc >= best_val_auc:
        best_val_auc = mean_auc
        torch.save(model.state_dict(), best_checkpoint_path)
        print(f"    [+] Checkpoint saved with AUROC: {best_val_auc:.4f}")


### Step 6: Multi-Label AUROC Evaluation (Model Card Table)
Computes independent AUROC for each of the 14 thoracic disease categories.


In [ ]:
model.load_state_dict(torch.load(best_checkpoint_path, map_location=device))
model.eval()

# Complete validation collection
val_logits_list, val_labels_list = [], []
with torch.no_grad():
    for imgs, labels, _ in val_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        val_logits_list.append(logits.cpu())
        val_labels_list.append(labels)

val_logits = torch.cat(val_logits_list, dim=0)
val_labels = torch.cat(val_labels_list, dim=0)
val_probs  = torch.sigmoid(val_logits).numpy()
val_targets = val_labels.numpy()

print(f"{'Pathology':<22} | {'Validation AUROC':<18} | {'Positives':<10}")
print("-" * 55)
class_aucs = {}
for i, disease in enumerate(DISEASE_CLASSES):
    y_true = val_targets[:, i]
    y_prob = val_probs[:, i]
    if np.sum(y_true) > 0:
        auc = roc_auc_score(y_true, y_prob)
        class_aucs[disease] = round(float(auc), 4)
        print(f"{disease:<22} | {auc:.4f}             | {int(np.sum(y_true)):<10}")
    else:
        class_aucs[disease] = 0.50

overall_mean_auc = np.mean(list(class_aucs.values()))
print("-" * 55)
print(f"{'OVERALL MEAN AUROC':<22} | {overall_mean_auc:.4f}")


### Step 7: Post-Hoc Confidence Calibration (Temperature Scaling)
Raw deep learning sigmoids are notoriously overconfident. We learn a positive temperature scalar $T_i$ for each disease using L-BFGS to minimize Expected Calibration Error (ECE).


In [ ]:
class ModelWithTemperature(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(num_classes))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        temp = torch.clamp(self.temperature, min=0.1, max=10.0)
        return logits / temp.unsqueeze(0)

    def calibrate(self, logits_val: torch.Tensor, labels_val: torch.Tensor, max_iter=50) -> Dict[str, float]:
        bce_criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.LBFGS([self.temperature], lr=0.05, max_iter=max_iter)

        def eval_loss():
            optimizer.zero_grad()
            scaled_logits = self.forward(logits_val)
            loss = bce_criterion(scaled_logits, labels_val)
            loss.backward()
            return loss

        optimizer.step(eval_loss)
        return {
            DISEASE_CLASSES[i]: round(float(self.temperature[i].item()), 4)
            for i in range(NUM_CLASSES)
        }

calibrator = ModelWithTemperature(num_classes=NUM_CLASSES)
learned_temps = calibrator.calibrate(val_logits, val_labels)

# Save temperature scaler
temp_path = "/kaggle/working/pretrained/temperature_scaler.json"
with open(temp_path, "w") as f:
    json.dump(learned_temps, f, indent=4)

print(f"[+] Learned Temperature Scaling Parameters saved to: {temp_path}")
for k, v in learned_temps.items():
    print(f"    - {k:<20}: T = {v:.4f}")


### Step 8: Optimal Operating Thresholds (Youden's J-Statistic)
Instead of naive 0.5 cutoffs, we calculate $J = \text{Sensitivity} + \text{Specificity} - 1$ per class to balance false alarms and missed critical findings.


In [ ]:
calibrated_val_probs = torch.sigmoid(calibrator(val_logits)).detach().numpy()

optimal_thresholds = {}
for i, disease in enumerate(DISEASE_CLASSES):
    y_true = val_targets[:, i]
    y_prob = calibrated_val_probs[:, i]

    if np.sum(y_true) == 0:
        optimal_thresholds[disease] = 0.50
        continue

    fpr, tpr, thresh = roc_curve(y_true, y_prob)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    best_thresh = float(thresh[best_idx])
    best_thresh = max(0.10, min(0.85, best_thresh))
    optimal_thresholds[disease] = round(best_thresh, 4)

thresh_path = "/kaggle/working/pretrained/optimal_thresholds.json"
with open(thresh_path, "w") as f:
    json.dump(optimal_thresholds, f, indent=4)

print(f"[+] Optimal Operating Thresholds saved to: {thresh_path}")
for k, v in optimal_thresholds.items():
    print(f"    - {k:<20}: Cutoff = {v:.4f}")


### Step 9: Grad-CAM++ BBox Benchmark against Ground Truth
We benchmark Grad-CAM++ against the 1,600 radiologist bounding boxes in `BBox_List_2017.csv` computing $IoBB$ (Intersection over Bounding Box Area) and $IoU$.


In [ ]:
class GradCAMPlusPlus:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self._forward_hook)
        target_layer.register_backward_hook(self._backward_hook)

    def _forward_hook(self, module, input, output):
        self.activations = output.detach()

    def _backward_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor: torch.Tensor, class_idx: int) -> np.ndarray:
        self.model.eval()
        self.model.zero_grad()

        output = self.model(input_tensor)
        target_score = output[0, class_idx]
        target_score.backward(retain_graph=True)

        grads = self.gradients[0]
        acts = self.activations[0]

        grads_power_2 = grads.pow(2)
        grads_power_3 = grads_power_2 * grads
        sum_acts = acts.sum(dim=(1, 2), keepdim=True)

        eps = 1e-7
        aij = grads_power_2 / (2 * grads_power_2 + sum_acts * grads_power_3 + eps)
        aij = torch.where(grads != 0, aij, torch.zeros_like(aij))

        weights = (aij * F.relu(grads)).sum(dim=(1, 2), keepdim=True)
        cam = (weights * acts).sum(dim=0)
        cam = F.relu(cam)

        cam_np = cam.cpu().numpy()
        cam_min, cam_max = cam_np.min(), cam_np.max()
        if cam_max > cam_min:
            cam_np = (cam_np - cam_min) / (cam_max - cam_min)
        else:
            cam_np = np.zeros_like(cam_np)
        return cam_np

# Attach Grad-CAM++ to DenseNet norm5 layer
target_layer = model.features.norm5
cam_engine = GradCAMPlusPlus(model, target_layer)

hits_iobb_10, hits_iobb_25, hits_iou_10 = 0, 0, 0
total_evaluated = 0

if not holdout_df.empty and bbox_df is not None:
    merged = pd.merge(bbox_df, holdout_df[["Image Index"]].drop_duplicates(), on="Image Index")
    for _, row in merged.head(150).iterrows():
        fname = row["Image Index"]
        disease = row["Finding Label"]
        if disease not in DISEASE_CLASSES: continue

        disease_idx = DISEASE_CLASSES.index(disease)
        img_path = image_lookup.get(fname, "")
        if not img_path or not os.path.exists(img_path): continue

        img = Image.open(img_path).convert("RGB").resize((224, 224))
        img_arr = np.array(img, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(img_arr).permute(2, 0, 1).float()
        for c in range(3):
            tensor[c] = (tensor[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
        tensor = tensor.unsqueeze(0).to(device)

        cam = cam_engine.generate(tensor, disease_idx)
        thresh = np.percentile(cam, 75)
        binary_mask = (cam >= thresh).astype(np.uint8)

        y_indices, x_indices = np.where(binary_mask > 0)
        if len(x_indices) == 0: continue

        scale = 1024.0 / 224.0
        pred_x1 = np.min(x_indices) * scale
        pred_x2 = np.max(x_indices) * scale
        pred_y1 = np.min(y_indices) * scale
        pred_y2 = np.max(y_indices) * scale

        gt_x1 = row["Bbox [x]"] if "Bbox [x]" in row else row["x"]
        gt_y1 = row["Bbox [y]"] if "Bbox [y]" in row else row["y"]
        gt_w  = row["Bbox [w]"] if "Bbox [w]" in row else row["w"]
        gt_h  = row["Bbox [h]"] if "Bbox [h]" in row else row["h"]
        gt_x2 = gt_x1 + gt_w
        gt_y2 = gt_y1 + gt_h

        inter_x1 = max(pred_x1, gt_x1)
        inter_y1 = max(pred_y1, gt_y1)
        inter_x2 = min(pred_x2, gt_x2)
        inter_y2 = min(pred_y2, gt_y2)

        inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
        pred_area  = (pred_x2 - pred_x1) * (pred_y2 - pred_y1)
        gt_area    = gt_w * gt_h
        union_area = pred_area + gt_area - inter_area

        iobb = inter_area / (pred_area + 1e-7)
        iou  = inter_area / (union_area + 1e-7)

        if iobb >= 0.10: hits_iobb_10 += 1
        if iobb >= 0.25: hits_iobb_25 += 1
        if iou  >= 0.10: hits_iou_10  += 1
        total_evaluated += 1

acc_iobb_10 = (hits_iobb_10 / max(1, total_evaluated)) * 100
acc_iobb_25 = (hits_iobb_25 / max(1, total_evaluated)) * 100
acc_iou_10  = (hits_iou_10  / max(1, total_evaluated)) * 100

print(f"[+] Localization Benchmark (N={total_evaluated} boxes):")
print(f"    - Accuracy (IoBB >= 0.10): {acc_iobb_10:.2f}%")
print(f"    - Accuracy (IoBB >= 0.25): {acc_iobb_25:.2f}%")
print(f"    - Accuracy (IoU  >= 0.10): {acc_iou_10:.2f}%")


### Step 10: Export Complete Production Artifacts
Saves the consolidated `validation_metrics.json` and prepares the files for download.


In [ ]:
metrics_summary = {
    "model_architecture": "DenseNet121",
    "input_resolution": "224x224",
    "training_sample_size": len(train_df),
    "validation_sample_size": len(val_df),
    "mean_validation_auroc": round(float(overall_mean_auc), 4),
    "class_wise_auroc": class_aucs,
    "temperature_scaling_parameters": learned_temps,
    "optimal_operating_thresholds": optimal_thresholds,
    "localization_benchmark": {
        "acc_iobb_10": round(float(acc_iobb_10), 2),
        "acc_iobb_25": round(float(acc_iobb_25), 2),
        "acc_iou_10":  round(float(acc_iou_10), 2),
        "total_evaluated_boxes": total_evaluated
    }
}

metrics_path = "/kaggle/working/pretrained/validation_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_summary, f, indent=4)

print("=" * 70)
print("VISIONCARE AI PIPELINE COMPLETE")
print("=" * 70)
print("Files ready in /kaggle/working/pretrained/:")
for f in os.listdir("/kaggle/working/pretrained"):
    fpath = os.path.join("/kaggle/working/pretrained", f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  • {f:<30} ({size_mb:.2f} MB)")

print("\n[*] Download these 4 files and place them into your local `visioncare-ai/pretrained/` folder!")
